# Chapter 4: Data Structures

This notebook covers the essential data structures used in scientific computing:

1. **Python Lists** — ordered, mutable sequences
2. **Stack** — Last-In-First-Out (LIFO)
3. **Queue** — First-In-First-Out (FIFO)
4. **Hash Map (Dictionary)** — O(1) key-value lookups
5. **NumPy Arrays** — fast vectorised numerical computing
6. **Pandas DataFrame** — tabular data analysis
7. **Multiprocessing** — parallelise CPU-bound work

In [1]:
import time
import sys
import numpy as np
import pandas as pd
from collections import deque, defaultdict
from multiprocessing import Pool, cpu_count
import matplotlib.pyplot as plt

---
## 1. Python Lists

A Python `list` is a **dynamic array**: elements are stored contiguously and accessed by integer index in O(1). Appending to the end is amortised O(1); inserting/deleting in the middle is O(n).

### 1.1 Creating Lists

In [2]:
# Three common ways to create lists
my_list = [1, 2, 3, 4, 5]
from_range = list(range(1, 6))
zeros = [0] * 10

print("Literal  :", my_list)
print("From range:", from_range)
print("Zeros     :", zeros)

Literal  : [1, 2, 3, 4, 5]
From range: [1, 2, 3, 4, 5]
Zeros     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### 1.2 Basic Operations

| Operation | Method | Time |
|---|---|---|
| Append to end | `append(x)` | O(1) amortised |
| Find index of value | `index(x)` | O(n) |
| Delete by index | `del lst[i]` | O(n) |

In [3]:
lst = [1, 2, 3, 4, 5]

# Append
lst.append(6)
print("After append:", lst)

# Search — returns first index where value == 5
print("Index of 5  :", lst.index(5))

# Delete by index
del lst[0]
print("After del[0]:", lst)

After append: [1, 2, 3, 4, 5, 6]
Index of 5  : 4
After del[0]: [2, 3, 4, 5, 6]


### 1.3 List Slicing

Syntax: `lst[start:stop:step]`  
Negative indices count from the end.

In [4]:
s = list(range(10))   # [0, 1, 2, …, 9]
print("Full list         :", s)
print("First 3  [0:3]    :", s[0:3])
print("Last 3   [-3:]    :", s[-3:])
print("Every 2nd [::2]   :", s[::2])
print("Reversed  [::-1]  :", s[::-1])

Full list         : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
First 3  [0:3]    : [0, 1, 2]
Last 3   [-3:]    : [7, 8, 9]
Every 2nd [::2]   : [0, 2, 4, 6, 8]
Reversed  [::-1]  : [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]


### 1.4 List Comprehensions (Prefer over explicit loops)

List comprehensions are more **readable** and often **faster** than equivalent `for` loops because the looping overhead is pushed into C.

In [5]:
numbers = list(range(1000))

# Traditional loop
def square_loop(nums):
    result = []
    for n in nums:
        result.append(n**2)
    return result

# List comprehension
def square_comp(nums):
    return [n**2 for n in nums]

# Comprehension with filter (only even numbers)
def square_even(nums):
    return [n**2 for n in nums if n % 2 == 0]

# Compare speed
t0 = time.perf_counter()
for _ in range(1000): square_loop(numbers)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(1000): square_comp(numbers)
t_comp = time.perf_counter() - t0

print(f"Loop         : {t_loop:.3f}s")
print(f"Comprehension: {t_comp:.3f}s  ({t_loop/t_comp:.1f}× faster)")
print("Even squares (first 10):", square_even(numbers)[:10])

Loop         : 0.033s
Comprehension: 0.029s  (1.2× faster)
Even squares (first 10): [0, 4, 16, 36, 64, 100, 144, 196, 256, 324]


### 1.5 Functional Utilities: map, filter, zip, enumerate

In [6]:
nums = [1, 2, 3, 4, 5]

# map — apply function to every element
squared = list(map(lambda x: x**2, nums))
print("map (squares):", squared)

# filter — keep elements satisfying a predicate
evens = list(filter(lambda x: x % 2 == 0, nums))
print("filter (even):", evens)

# any / all
print("any even?  ", any(x % 2 == 0 for x in nums))   # True
print("all positive?", all(x > 0 for x in nums))       # True

# zip — pair elements from two lists
letters = ['a', 'b', 'c', 'd', 'e']
pairs = list(zip(nums, letters))
print("zip pairs  :", pairs)

# enumerate — get (index, value) pairs
for idx, val in enumerate(letters[:3]):
    print(f"  index {idx}: {val}")

map (squares): [1, 4, 9, 16, 25]
filter (even): [2, 4]
any even?   True
all positive? True
zip pairs  : [(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd'), (5, 'e')]
  index 0: a
  index 1: b
  index 2: c


---
## 2. Stack (LIFO)

A **stack** follows **Last-In-First-Out**: the most recently added element is removed first — like a pile of plates.

In Python, a plain `list` works as a stack:
- `push` → `list.append()`  
- `pop`  → `list.pop()`  
- Both are O(1).

In [7]:
# Stack using a Python list
stack = []

# Push elements
for val in [1, 3, 2, 5, 4]:
    stack.append(val)
print("Stack after pushes:", stack)
print("Top (peek)        :", stack[-1])

# Pop — removes and returns the top element
top = stack.pop()
print(f"Popped            : {top}")
print("Stack after pop   :", stack)

print("Size              :", len(stack))
print("Empty?            :", len(stack) == 0)

Stack after pushes: [1, 3, 2, 5, 4]
Top (peek)        : 4
Popped            : 4
Stack after pop   : [1, 3, 2, 5]
Size              : 4
Empty?            : False


**Classic use-case**: matching brackets. A stack lets us track open brackets and check that each closing bracket matches the most recent open one.

In [8]:
def is_balanced(s: str) -> bool:
    """Return True if all brackets in s are properly matched."""
    match = {')': '(', ']': '[', '}': '{'}
    stack = []
    for ch in s:
        if ch in '([{':
            stack.append(ch)
        elif ch in ')]}':
            if not stack or stack[-1] != match[ch]:
                return False
            stack.pop()
    return len(stack) == 0

tests = ["([])", "([)]", "{[()]}", "(((", ""]
for t in tests:
    print(f"  {t!r:12} → {is_balanced(t)}")

  '([])'       → True
  '([)]'       → False
  '{[()]}'     → True
  '((('        → False
  ''           → True


---
## 3. Queue (FIFO)

A **queue** follows **First-In-First-Out**: the first element added is the first to be removed — like a checkout line.

Use `collections.deque` (double-ended queue): both `append` and `popleft` are O(1). (Using a plain list is **not** recommended because `list.pop(0)` is O(n).)

In [9]:
from collections import deque

que = deque()

# Enqueue (add to tail)
for val in [1, 3, 2, 5, 4]:
    que.append(val)
print("Queue after enqueues:", list(que))
print("Front (peek)        :", que[0])

# Dequeue (remove from front)
front = que.popleft()
print(f"Dequeued            : {front}")
print("Queue after dequeue :", list(que))

print("Size                :", len(que))
print("Empty?              :", len(que) == 0)

Queue after enqueues: [1, 3, 2, 5, 4]
Front (peek)        : 1
Dequeued            : 1
Queue after dequeue : [3, 2, 5, 4]
Size                : 4
Empty?              : False


**Why queue for BFS?** A queue ensures we always process nodes in the order we discovered them, guaranteeing level-by-level exploration.

In [10]:
# Simple BFS on a tree represented as {node: [children]}
def bfs_level_order(tree: dict, root) -> list:
    """Return nodes visited in BFS (level) order."""
    order, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        order.append(node)
        for child in tree.get(node, []):
            queue.append(child)
    return order

tree = {1: [2, 3], 2: [4, 5], 3: [6]}
print("BFS order:", bfs_level_order(tree, 1))   # [1, 2, 3, 4, 5, 6]

BFS order: [1, 2, 3, 4, 5, 6]


---
## 4. Hash Map (Dictionary)

A **hash map** stores key-value pairs. Under the hood it uses a hash function to map keys to array slots, giving **O(1) average** insert / delete / search.

| Operation | List | Dict |
|---|---|---|
| Search | O(n) | **O(1)** |
| Insert | O(1) | **O(1)** |
| Delete | O(n) | **O(1)** |

In [11]:
# Creating a dictionary
hmap = {'apple': 5, 'banana': 3, 'orange': 2}

# Dictionary comprehension
squared_dict = {k: v**2 for k, v in hmap.items()}
print("Original :", hmap)
print("Squared  :", squared_dict)

# Traversal
print("\nKey → Value:")
for key, value in hmap.items():
    print(f"  {key} → {value}")

# defaultdict: no KeyError for missing keys
counts = defaultdict(int)
for word in ['apple', 'banana', 'apple', 'orange', 'apple']:
    counts[word] += 1
print("\nWord counts:", dict(counts))

Original : {'apple': 5, 'banana': 3, 'orange': 2}
Squared  : {'apple': 25, 'banana': 9, 'orange': 4}

Key → Value:
  apple → 5
  banana → 3
  orange → 2

Word counts: {'apple': 3, 'banana': 1, 'orange': 1}


### 4.1 Hash Optimization: Two-Sum Problem

**Problem**: Given an integer array `nums` and target `t`, find two indices i, j such that `nums[i] + nums[j] == t`.

- **Brute force**: two nested loops → O(n²)
- **Hash table**: single pass → O(n)  ← **trade space for time**

In [12]:
def two_sum_brute(nums: list, target: int) -> list:
    """O(n²) time, O(1) space — check every pair."""
    for i in range(len(nums) - 1):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return []

def two_sum_hash(nums: list, target: int) -> list:
    """O(n) time, O(n) space — hash table for instant complement lookup."""
    seen = {}  # value -> index
    for i, val in enumerate(nums):
        complement = target - val
        if complement in seen:
            return [seen[complement], i]
        seen[val] = i
    return []

nums = [2, 7, 11, 15]
target = 9
print("Brute force:", two_sum_brute(nums, target))   # [0, 1]
print("Hash table :", two_sum_hash(nums, target))    # [0, 1]

# Performance comparison
import random
big = list(range(10_000))
big_target = 19_997

t0 = time.perf_counter(); two_sum_brute(big, big_target); t_b = time.perf_counter() - t0
t0 = time.perf_counter(); two_sum_hash(big, big_target);  t_h = time.perf_counter() - t0

print(f"\nn=10,000: brute={t_b:.4f}s  hash={t_h:.6f}s  ({t_b/max(t_h,1e-9):.0f}× speedup)")

Brute force: [0, 1]
Hash table : [0, 1]



n=10,000: brute=1.7859s  hash=0.000736s  (2426× speedup)


---
## 5. NumPy Arrays

NumPy provides **fixed-type, multi-dimensional arrays** backed by optimised BLAS/LAPACK C code.  
Key advantages over Python lists:
- **Vectorised operations** — no Python loop overhead
- **Compact memory** — homogeneous dtype, no Python object overhead
- **Rich linear algebra** — built-in matrix ops

### 5.1 Creating Arrays

In [13]:
# From a Python list
a = np.array([1, 2, 3, 4, 5])
print("Array  :", a)
print("Shape  :", a.shape)
print("dtype  :", a.dtype)

# Useful constructors
zeros   = np.zeros(5)
ones    = np.ones((2, 3))
rng     = np.arange(0, 10, 2)
linsp   = np.linspace(0, 1, 5)
rand    = np.random.rand(3, 3)

print("\narange :", rng)
print("linspace:", linsp)
print("2D ones :\n", ones)

Array  : [1 2 3 4 5]
Shape  : (5,)
dtype  : int64

arange : [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
2D ones :
 [[1. 1. 1.]
 [1. 1. 1.]]


### 5.2 Vectorised Operations (No Loops!)

In [14]:
a = np.array([1, 2, 3, 4, 5])

print("Element-wise square:", a**2)
print("Mean              :", a.mean())
print("Sum               :", a.sum())
print("Max               :", a.max())

# Boolean masking — no explicit loop
print("Elements > 3      :", a[a > 3])
print("Even elements     :", a[a % 2 == 0])

# np.where — index of matching elements
print("Indices of evens  :", np.where(a % 2 == 0)[0])

Element-wise square: [ 1  4  9 16 25]
Mean              : 3.0
Sum               : 15
Max               : 5
Elements > 3      : [4 5]
Even elements     : [2 4]
Indices of evens  : [1 3]


### 5.3 Array Slicing

In [15]:
arr = np.arange(10)
print("Full array  :", arr)
print("First 3     :", arr[0:3])
print("Last 3      :", arr[-3:])
print("Every 2nd   :", arr[::2])

# 2-D slicing
mat = np.array([[1, 2, 3],
                [4, 5, 6],
                [7, 8, 9]])
print("\n2D matrix:\n", mat)
print("First 2 rows         :\n", mat[0:2, :])
print("Last 2 columns       :\n", mat[:, 1:])
print("Centre element (1,1) :",  mat[1, 1])

Full array  : [0 1 2 3 4 5 6 7 8 9]
First 3     : [0 1 2]
Last 3      : [7 8 9]
Every 2nd   : [0 2 4 6 8]

2D matrix:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
First 2 rows         :
 [[1 2 3]
 [4 5 6]]
Last 2 columns       :
 [[2 3]
 [5 6]
 [8 9]]
Centre element (1,1) : 5


### 5.4 Matrix Operations

In [16]:
X = np.arange(12).reshape(3, 4).astype(float)
print("X:\n", X)

# Column means and centering
col_means = X.mean(axis=0)
X_centered = X - col_means
print("\nColumn means :", col_means)
print("Centered X   :\n", X_centered)

X:
 [[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]]

Column means : [4. 5. 6. 7.]
Centered X   :
 [[-4. -4. -4. -4.]
 [ 0.  0.  0.  0.]
 [ 4.  4.  4.  4.]]


### 5.5 NumPy vs Python List: Speed Comparison

In [17]:
n = 1_000_000
lst = list(range(n))
arr = np.arange(n)

# Sum
t0 = time.perf_counter(); sum(lst);      t_list = time.perf_counter() - t0
t0 = time.perf_counter(); arr.sum();     t_np   = time.perf_counter() - t0
print(f"Sum  n={n}: list={t_list:.4f}s  numpy={t_np:.6f}s  ({t_list/max(t_np,1e-9):.0f}× faster)")

# Memory
list_mem = sys.getsizeof(lst) + sum(sys.getsizeof(x) for x in range(min(100,n))) * n // 100
np_mem   = arr.nbytes
print(f"Memory n={n}: numpy={np_mem/1e6:.1f} MB  (list is ~{list_mem/1e6:.0f}+ MB)")

Sum  n=1000000: list=0.0058s  numpy=0.000763s  (8× faster)
Memory n=1000000: numpy=8.0 MB  (list is ~36+ MB)


---
## 6. Pandas DataFrame

Pandas `DataFrame` is a labelled 2-D table — think of it as a spreadsheet in Python. It is built on top of NumPy and is the standard tool for data wrangling.

### 6.1 Creating DataFrames

In [18]:
df = pd.DataFrame({
    'Name':  ['Alice', 'Bob', 'Carol', 'Dave'],
    'Age':   [25, 30, 35, 28],
    'Score': [88.5, 92.0, 79.5, 95.0]
})
print(df)
print("\nShape:", df.shape)
print("dtypes:\n", df.dtypes)

    Name  Age  Score
0  Alice   25   88.5
1    Bob   30   92.0
2  Carol   35   79.5
3   Dave   28   95.0

Shape: (4, 3)
dtypes:
 Name      object
Age        int64
Score    float64
dtype: object


### 6.2 Selecting and Filtering

In [19]:
# Select a column
print("Scores:\n", df['Score'].values)

# Boolean filtering
high_scorers = df[df['Score'] > 90]
print("\nHigh scorers:\n", high_scorers)

# iloc — integer-position based
print("\nFirst 2 rows:\n", df.iloc[0:2])

# loc — label based
print("\nNames and Scores:\n", df.loc[:, ['Name', 'Score']])

Scores:
 [88.5 92.  79.5 95. ]

High scorers:
    Name  Age  Score
1   Bob   30   92.0
3  Dave   28   95.0

First 2 rows:
     Name  Age  Score
0  Alice   25   88.5
1    Bob   30   92.0

Names and Scores:
     Name  Score
0  Alice   88.5
1    Bob   92.0
2  Carol   79.5
3   Dave   95.0


### 6.3 Apply and Aggregation

In [20]:
# Apply a function column-wise
print("Column means:\n", df[['Age', 'Score']].apply(np.mean))

# Row-wise
print("\nRow means:\n", df[['Age', 'Score']].apply(np.mean, axis=1))

# Describe — quick stats
print("\nDescriptive stats:\n", df[['Age', 'Score']].describe())

Column means:
 Age      29.50
Score    88.75
dtype: float64

Row means:
 0    56.75
1    61.00
2    57.25
3    61.50
dtype: float64

Descriptive stats:
              Age      Score
count   4.000000   4.000000
mean   29.500000  88.750000
std     4.203173   6.714412
min    25.000000  79.500000
25%    27.250000  86.250000
50%    29.000000  90.250000
75%    31.250000  92.750000
max    35.000000  95.000000


### 6.4 Drop Rows / Columns

In [21]:
print("Drop 'Age' column:\n", df.drop(columns=['Age']))
print("\nDrop rows 0 and 2:\n", df.drop([0, 2]))

Drop 'Age' column:
     Name  Score
0  Alice   88.5
1    Bob   92.0
2  Carol   79.5
3   Dave   95.0

Drop rows 0 and 2:
    Name  Age  Score
1   Bob   30   92.0
3  Dave   28   95.0


---
## 7. Multiprocessing

Python's Global Interpreter Lock (GIL) prevents true multi-threading for CPU-bound work. The `multiprocessing` module sidesteps the GIL by spawning **separate processes** — each with its own Python interpreter and memory space.

**Use it when**: the task is CPU-intensive, the data can be split into independent chunks.

In [22]:
from multiprocessing.pool import ThreadPool   # thread-based pool — works inside Jupyter

def square_root_sum(n: int) -> float:
    """CPU-intensive: sum of square roots from 0 to n-1."""
    return sum(i**0.5 for i in range(n))

chunks = [200_000] * 8    # 8 independent chunks

# Sequential baseline
t0 = time.perf_counter()
results_seq = [square_root_sum(c) for c in chunks]
t_seq = time.perf_counter() - t0

# ThreadPool — same Pool API, runs inside Jupyter notebooks
# Note: For a standalone .py script use multiprocessing.Pool for true
# process-based parallelism that bypasses the GIL.
num_threads = max(1, cpu_count() - 1)
t0 = time.perf_counter()
with ThreadPool(processes=num_threads) as pool:
    results_par = pool.map(square_root_sum, chunks)
t_par = time.perf_counter() - t0

print(f"CPUs available : {cpu_count()}")
print(f"Sequential     : {t_seq:.3f}s")
print(f"ThreadPool     : {t_par:.3f}s  ({t_seq/max(t_par,1e-9):.1f}× speedup)")
print("Results match  :", np.allclose(results_seq, results_par))


CPUs available : 8
Sequential     : 0.111s
ThreadPool     : 0.125s  (0.9× speedup)
Results match  : True


### 7.1 Multiprocessing Template

In [23]:
# General multiprocessing template
# In a standalone .py script, replace ThreadPool with multiprocessing.Pool
# for true CPU-level parallelism across separate processes.

def process_item(item):
    """Worker function applied independently to each element."""
    return item * 2   # replace with your real computation

def parallel_processing(data_list):
    num_workers = max(1, cpu_count() - 1)
    with ThreadPool(processes=num_workers) as pool:
        results = pool.map(process_item, data_list)
    return results

data = list(range(10))
print("Input  :", data)
print("Output :", parallel_processing(data))
print()
print("Key tips:")
print("  - Keep worker functions simple and stateless")
print("  - Minimise data transfer between workers")
print("  - Use multiprocessing.Pool (not ThreadPool) for CPU-bound tasks in .py scripts")


Input  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Output : [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

Key tips:
  - Keep worker functions simple and stateless
  - Minimise data transfer between workers
  - Use multiprocessing.Pool (not ThreadPool) for CPU-bound tasks in .py scripts


---
## 8. Summary: When to Use Which Data Structure

| Data Structure | Best For | Key Complexity |
|---|---|---|
| **List** | Ordered data, integer indexing, mixed types | O(1) append, O(n) search |
| **Stack** | Undo / bracket matching / DFS | O(1) push/pop |
| **Queue** | BFS / task scheduling / streaming | O(1) enqueue/dequeue |
| **Dict** | Fast lookup by key, counting, grouping | O(1) avg |
| **NumPy array** | Numerical / matrix computation | Vectorised (C speed) |
| **DataFrame** | Tabular data analysis | Depends on operation |
| **Multiprocessing** | CPU-bound parallelism | ~linear speedup with cores |